In [1]:
import pandas as pd
import gc

#### prep bus and mrt data for the datasets

In [2]:
bus_data = pd.read_csv('../../data/A0008D - v_q_bus_stop (full).csv')
bus_location_data = pd.read_csv('../../data/correct_bus_location.csv')
bus_additional_data = pd.read_csv('../../data/A0008D - v_q_vls_marker (full).csv')

In [3]:
bus_consolidated = bus_data.merge(
   bus_location_data,
   how='outer',
   left_on='BUS_STOP_CD',
   right_on='BusStopCode'
)


bus_needed = bus_consolidated[[
   "BUS_STOP_CD",
   "BUS_STOP_NAM",
   "MRK_ID_NUM",
   "BusStopCode",
   "Description",
   "Latitude",
   "Longitude"
]].copy()


bus_needed = bus_needed.rename(columns={
   "BUS_STOP_NAM": "STATION_NAME",
   "Latitude": "LATITUDE",
   "Longitude": "LONGITUDE"
})


bus_needed = bus_needed.rename(columns={
   "BUS_STOP_NAM": "STATION_NAME",
   "Description": "LTA_DESCRIPTION",
   "Latitude": "LATITUDE",
   "Longitude": "LONGITUDE"
})
bus_needed["FINAL_STOP_CODE"] = bus_needed["BUS_STOP_CD"].fillna(bus_needed["BusStopCode"])
bus_needed["STATION_NAME"] = bus_needed["STATION_NAME"].fillna(bus_needed["LTA_DESCRIPTION"])
bus_needed['Travel_Type'] = 'BUS'


bus_needed_final = bus_needed[[
   "FINAL_STOP_CODE",
   "STATION_NAME",
   "MRK_ID_NUM",
   "LATITUDE",
   "LONGITUDE",
   "Travel_Type"
]].copy()


bus_additional_data["LATITUDE"] = bus_additional_data["LATITUDE_VAL"] / 3600000
bus_additional_data["LONGITUDE"] = bus_additional_data["LONGITUDE_VAL"] / 3600000


bus_additional_data_clean = bus_additional_data[[
   "MRK_ID_NUM",
   "LATITUDE",
   "LONGITUDE"
]].copy()


bus_needed_final = bus_needed_final.merge(
   bus_additional_data_clean,
   on="MRK_ID_NUM",
   how="left",
   suffixes=("", "_additional")
)


bus_needed_final["LATITUDE"] = bus_needed_final["LATITUDE"].fillna(
   bus_needed_final["LATITUDE_additional"]
)


bus_needed_final["LONGITUDE"] = bus_needed_final["LONGITUDE"].fillna(
   bus_needed_final["LONGITUDE_additional"]
)


bus_needed_final = bus_needed_final.drop(columns=[
   "LATITUDE_additional",
   "LONGITUDE_additional"
])


print("Remaining missing coords:",
     bus_needed_final["LATITUDE"].isna().sum())

Remaining missing coords: 0


In [4]:
train_data = pd.read_csv('../../data/mrt_station_coordinates.csv')
train_needed = train_data[['NODE_NAME', 'NODE_MRK_ID', 'LATITUDE', 'LONGITUDE']]
train_needed = train_needed.rename(columns={
   'NODE_NAME': 'STATION_NAME',
   'NODE_MRK_ID': 'MRK_ID_NUM'
})
train_needed['Travel_Type'] = 'TRAIN'
#train_needed.head(5)

In [5]:
bus_needed_final = bus_needed_final[[
   "STATION_NAME",
   "MRK_ID_NUM",
   "LATITUDE",
   "LONGITUDE",
   "Travel_Type"
]]


consolidated_stations = pd.concat(
   [bus_needed_final, train_needed],
   axis=0,
   ignore_index=True
)


consolidated_stations.tail(5)

,STATION_NAME,MRK_ID_NUM,LATITUDE,LONGITUDE,Travel_Type
6271,Marine Parade,427.0,1.302865,103.905508,TRAIN
6272,Marine Terrace,428.0,1.306786,103.915316,TRAIN
6273,Siglap,429.0,1.309877,103.929879,TRAIN
6274,Bayshore,430.0,1.312841,103.941597,TRAIN
6275,Hume,336.0,1.354511,103.769104,TRAIN


In [ ]:
# memory checkpoint
# %who
# del bus_additional_data, bus_additional_data_clean,bus_consolidated, bus_data, bus_location_data, bus_needed, bus_needed_final, train_data, train_needed

bus_additional_data	 bus_additional_data_clean	 bus_consolidated	 bus_data	 bus_location_data	 bus_needed	 bus_needed_final	 consolidated_stations	 gc	 
pd	 train_data	 train_needed	 


#### for pt_ride

In [8]:
df = pd.read_csv('../../data/A0008D - v_pt_ride_all (full).csv')

In [10]:
df["BIZ_DT"].head()

0    2025-02-22
1    2025-02-22
2    2025-02-22
3    2025-02-22
4    2025-02-22
Name: BIZ_DT, dtype: object

# @ LK below is date filtering

In [ ]:
# # memory checkpoint - Filtering first
# df = df[df["ENTRY_DT"].str.startswith("2025-02-12")]
# print(df.shape)
# gc.collect()

(3820455, 21)


31

In [14]:
df['BIZ_DT'] = pd.to_datetime(df['BIZ_DT'])
df['ENTRY_DT'] = pd.to_datetime(df['ENTRY_DT'])
df['EXIT_DT'] = pd.to_datetime(df['EXIT_DT'])
df['ENTRY_TM'] = pd.to_datetime(
   df['ENTRY_DT'].astype(str) + ' ' + df['ENTRY_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
df['EXIT_TM'] = pd.to_datetime(
   df['EXIT_DT'].astype(str) + ' ' + df['EXIT_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)


In [8]:
df.dtypes

BIZ_DT                datetime64[us]
BUS_SVC_NUM                  float64
CRD_NUM                        int64
DEST_LOC_ID_NUM                int64
ENTRY_DT              datetime64[us]
ENTRY_TM              datetime64[us]
EXIT_DT               datetime64[us]
EXIT_TM               datetime64[us]
JRNY_ID_NUM                    int64
ORIG_LOC_ID_NUM                int64
PATRON_CATG_ID_NUM             int64
PAY_CD                         int64
RIDE_DISC_AMT                float64
RIDE_DIST_KM_CNT             float64
RIDE_FARE_AMT                float64
RIDE_ID_NUM                    int64
RIDE_MIN_CNT                 float64
RIDE_ST_CD                     int64
SVC_GRADE_ID_NUM               int64
TKT_TYP_CD                     int64
TRNSPT_MODE_CD                 int64
dtype: object

In [15]:
df2 = df[['BUS_SVC_NUM', 'CRD_NUM', 'DEST_LOC_ID_NUM', 'ENTRY_DT',
      'ENTRY_TM', 'EXIT_DT', 'EXIT_TM', 'JRNY_ID_NUM', 'ORIG_LOC_ID_NUM', 'RIDE_DISC_AMT', 'RIDE_DIST_KM_CNT',
      'RIDE_FARE_AMT', 'RIDE_ID_NUM', 'RIDE_MIN_CNT','PATRON_CATG_ID_NUM','TRNSPT_MODE_CD']]
#df2.head()

In [16]:
df2 = df2.merge(consolidated_stations, how = 'left', left_on= 'DEST_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')
#df2.head(5)

In [17]:
df2.rename(columns={
   'STATION_NAME': 'DEST_STATION_NAME',
   'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
   'LATITUDE': 'DEST_LATITUDE',
   'LONGITUDE': 'DEST_LONGITUDE',
   'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [18]:
df2 = df2.merge(consolidated_stations, how = 'left', left_on= 'ORIG_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [19]:
df2.rename(columns={
   'STATION_NAME': 'ORIG_STATION_NAME',
   'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
   'LATITUDE': 'ORIG_LATITUDE',
   'LONGITUDE': 'ORIG_LONGITUDE',
   'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)
#df2.head(5)

In [20]:
df2.shape

(3820455, 26)

In [ ]:
# # clear memory checkpoint
# %who
# del df

consolidated_stations	 df	 df2	 gc	 pd	 


### for abt_ride

In [23]:
abt = pd.read_csv('../../data/A0008D - v_abt_pt_ride (full).csv')

C:\Users\shaog\AppData\Local\Temp\ipykernel_13736\2821325984.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  abt = pd.read_csv('../../data/A0008D - v_abt_pt_ride (full).csv')


In [ ]:
print(abt.shape)

In [ ]:
# abt["ENTRY_DT"].isna().sum()
# abt[abt["ENTRY_DT"].isna()].head(20)


In [35]:
gc.collect()

7

In [ ]:
# # memory checkpoint - Filtering first
# abt = abt[abt["ENTRY_DT"].str.startswith("2025-02-12", na=False)]
# print(abt.shape)
# gc.collect()

(3655600, 21)


0

In [38]:
abt['BIZ_DT'] = pd.to_datetime(abt['BIZ_DT'])
abt['ENTRY_DT'] = pd.to_datetime(abt['ENTRY_DT'])
abt['EXIT_DT'] = pd.to_datetime(abt['EXIT_DT'])
abt['ENTRY_TM'] = pd.to_datetime(
   abt['ENTRY_DT'].astype(str) + ' ' + abt['ENTRY_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
abt['EXIT_TM'] = pd.to_datetime(
   abt['EXIT_DT'].astype(str) + ' ' + abt['EXIT_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)

C:\Users\shaog\AppData\Local\Temp\ipykernel_13736\1923178911.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  abt['BIZ_DT'] = pd.to_datetime(abt['BIZ_DT'])
C:\Users\shaog\AppData\Local\Temp\ipykernel_13736\1923178911.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  abt['ENTRY_DT'] = pd.to_datetime(abt['ENTRY_DT'])
C:\Users\shaog\AppData\Local\Temp\ipykernel_13736\1923178911.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_

In [39]:
abt.dtypes

BIZ_DT                datetime64[ns]
BUS_SVC_NUM                  float64
CRD_NUM                       object
DEST_LOC_ID_NUM              float64
ENTRY_DT              datetime64[ns]
ENTRY_TM              datetime64[ns]
EXIT_DT               datetime64[ns]
EXIT_TM               datetime64[ns]
JRNY_ID_NUM                   object
ORIG_LOC_ID_NUM              float64
PATRON_CATG_ID_NUM             int64
PAY_CD                         int64
RIDE_DISC_AMT                float64
RIDE_DIST_KM_CNT             float64
RIDE_FARE_AMT                float64
RIDE_ID_NUM                   object
RIDE_MIN_CNT                 float64
RIDE_ST_CD                     int64
SVC_GRADE_ID_NUM             float64
TKT_TYP_CD                     int64
TRNSPT_MODE_CD                 int64
dtype: object

In [40]:
abt2 = abt[['BUS_SVC_NUM', 'CRD_NUM', 'DEST_LOC_ID_NUM', 'ENTRY_DT',
      'ENTRY_TM', 'EXIT_DT', 'EXIT_TM', 'JRNY_ID_NUM', 'ORIG_LOC_ID_NUM', 'RIDE_DISC_AMT', 'RIDE_DIST_KM_CNT',
      'RIDE_FARE_AMT', 'RIDE_ID_NUM', 'RIDE_MIN_CNT','PATRON_CATG_ID_NUM','TRNSPT_MODE_CD']]
#abt2.head()

In [42]:
# memory checkpoint
%who
del abt

abt	 abt2	 consolidated_stations	 df2	 gc	 pd	 


In [41]:
abt2 = abt2.merge(consolidated_stations, how = 'left', left_on= 'DEST_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')
#abt2.head(5)

In [43]:
abt2.rename(columns={
   'STATION_NAME': 'DEST_STATION_NAME',
   'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
   'LATITUDE': 'DEST_LATITUDE',
   'LONGITUDE': 'DEST_LONGITUDE',
   'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [44]:
abt2 = abt2.merge(consolidated_stations, how = 'left', left_on= 'ORIG_LOC_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [45]:
abt2.rename(columns={
   'STATION_NAME': 'ORIG_STATION_NAME',
   'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
   'LATITUDE': 'ORIG_LATITUDE',
   'LONGITUDE': 'ORIG_LONGITUDE',
   'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)
#abt2.head(5)

In [46]:
abt2.shape

(3662776, 26)

### Combine pt_ride and abt_ride

In [47]:
df2['CRD_NUM'] = df2['CRD_NUM'].astype('object')


In [26]:
df2.dtypes

BUS_SVC_NUM                  float64
CRD_NUM                       object
DEST_LOC_ID_NUM                int64
ENTRY_DT              datetime64[us]
ENTRY_TM              datetime64[us]
EXIT_DT               datetime64[us]
EXIT_TM               datetime64[us]
JRNY_ID_NUM                    int64
ORIG_LOC_ID_NUM                int64
RIDE_DISC_AMT                float64
RIDE_DIST_KM_CNT             float64
RIDE_FARE_AMT                float64
RIDE_ID_NUM                    int64
RIDE_MIN_CNT                 float64
PATRON_CATG_ID_NUM             int64
TRNSPT_MODE_CD                 int64
DEST_STATION_NAME                str
DEST_MRK_ID_NUM              float64
DEST_LATITUDE                float64
DEST_LONGITUDE               float64
DEST_Travel_Type                 str
ORIG_STATION_NAME                str
ORIG_MRK_ID_NUM              float64
ORIG_LATITUDE                float64
ORIG_LONGITUDE               float64
ORIG_Travel_Type                 str
dtype: object

In [48]:
df2 = pd.concat(
   [df2, abt2],
   axis=0,
   ignore_index=True
)


#df2.tail(5)

In [49]:
df2.shape

(7483231, 26)

In [51]:
df2.head()

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,...,DEST_STATION_NAME,DEST_MRK_ID_NUM,DEST_LATITUDE,DEST_LONGITUDE,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type
0,4.0,2190003183153,4549.0,2025-02-12,2025-02-12 07:06:55,2025-02-12,2025-02-12 07:22:08,110713962196,5622.0,0.0,...,Blk 233,4549.0,1.356603,103.947669,BUS,Aft Ballota Pk,5622.0,1.359266,103.968408,BUS
1,93.0,230008092904,4416.0,2025-02-12,2025-02-12 15:46:13,2025-02-12,2025-02-12 15:54:30,110713963174,4429.0,0.0,...,Blk 637,4416.0,1.330734,103.903732,BUS,Opp SCDF HQ,4429.0,1.334978,103.892387,BUS
2,293.0,230006064637,4512.0,2025-02-12,2025-02-12 19:10:55,2025-02-12,2025-02-12 19:18:57,110713963376,4481.0,0.0,...,Poi Ching Sch,4512.0,1.357755,103.936272,BUS,Blk 827,4481.0,1.349771,103.934047,BUS
3,985.0,240002874005,2770.0,2025-02-12,2025-02-12 06:40:53,2025-02-12,2025-02-12 06:47:19,110713964613,2754.0,0.0,...,Blk 419,2770.0,1.362596,103.746867,BUS,Aft Bt Batok Stn/Blk 628,2754.0,1.351623,103.749934,BUS
4,354.0,230001934402,4114.0,2025-02-12,2025-02-12 12:29:27,2025-02-12,2025-02-12 12:35:35,110713965235,5686.0,0.0,...,Pasir Ris Int Alighting,4114.0,1.373696,103.948450,BUS,Opp Lighthouse,5686.0,1.378021,103.959366,BUS


In [29]:
df2.dtypes

BUS_SVC_NUM                  float64
CRD_NUM                       object
DEST_LOC_ID_NUM              float64
ENTRY_DT              datetime64[us]
ENTRY_TM              datetime64[us]
EXIT_DT               datetime64[us]
EXIT_TM               datetime64[us]
JRNY_ID_NUM                   object
ORIG_LOC_ID_NUM              float64
RIDE_DISC_AMT                float64
RIDE_DIST_KM_CNT             float64
RIDE_FARE_AMT                float64
RIDE_ID_NUM                   object
RIDE_MIN_CNT                 float64
PATRON_CATG_ID_NUM             int64
TRNSPT_MODE_CD                 int64
DEST_STATION_NAME                str
DEST_MRK_ID_NUM              float64
DEST_LATITUDE                float64
DEST_LONGITUDE               float64
DEST_Travel_Type                 str
ORIG_STATION_NAME                str
ORIG_MRK_ID_NUM              float64
ORIG_LATITUDE                float64
ORIG_LONGITUDE               float64
ORIG_Travel_Type                 str
dtype: object

In [52]:
df2.to_pickle('df2.pkl')
print('Saved df2.pkl')

Saved df2.pkl


In [32]:
# To load the dataframe in another file later
# df2 = pd.read_pickle('df2.pkl')

### Add walking distance to df2, dataset used for modelling is df3

In [53]:
df3 = df2[df2["ENTRY_DT"].dt.date == pd.Timestamp("2025-02-12").date()]
df3 = df3.sort_values(["CRD_NUM", "ENTRY_DT", "ENTRY_TM"], ascending= [True, True, True]).reset_index(drop= True)

In [ ]:
# clear memory checkpoint
# %who
# del df2, abt2
# gc.collect()

In [58]:
# Onemap API does not allow dates more than a year
df3["ENTRY_DT"] = pd.Timestamp("10-04-2025")
df3["EXIT_DT"] = pd.Timestamp("10-04-2025")
df3["ENTRY_TM"] = df3["ENTRY_TM"].dt.strftime("%H:%M:%S")
df3["EXIT_TM"] = df3["EXIT_TM"].dt.strftime("%H:%M:%S")
df3["ENTRY_DT"] = df3["ENTRY_DT"].dt.strftime("%m-%d-%Y")
df3["EXIT_DT"] = df3["EXIT_DT"].dt.strftime("%m-%d-%Y")

In [59]:
df3.shape

(7483231, 26)

In [60]:
df3["next_orig_lat"] = df3.groupby("CRD_NUM")["ORIG_LATITUDE"].shift(-1)
df3["next_orig_lon"] = df3.groupby("CRD_NUM")["ORIG_LONGITUDE"].shift(-1)
df3["next_orig_station"] = df3.groupby("CRD_NUM")["ORIG_STATION_NAME"].shift(-1)

In [61]:
walk_cache = pd.read_csv('../../data/walk_distance_cache.csv')

In [62]:
df3["pair_key"] = (
  df3["DEST_LATITUDE"].astype(str) + "_" +
  df3["DEST_LONGITUDE"].astype(str) + "_" +
  df3["next_orig_lat"].astype(str) + "_" +
  df3["next_orig_lon"].astype(str)
)
pairs = df3[
   [
       "pair_key",
       "DEST_STATION_NAME",
       "next_orig_station",
       "DEST_LATITUDE",
       "DEST_LONGITUDE",
       "next_orig_lat",
       "next_orig_lon"
   ]
].dropna(subset=[
   "DEST_LATITUDE",
   "DEST_LONGITUDE",
   "next_orig_lat",
   "next_orig_lon"
]).drop_duplicates("pair_key").copy()


In [63]:
df3 = (
   df3.drop(columns=["pair_key"], errors="ignore")
   .merge(
       walk_cache.drop(columns=["pair_key"], errors="ignore")[[
           "DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon", "walk_distance"
       ]].drop_duplicates(
           subset=["DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon"]
       ),
       on=["DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon"],
       how="left"
   )
)

In [64]:
gc.collect()

0

In [65]:
# restore the actual filtered business date
real_date = pd.Timestamp("2025-02-12")


df3["ENTRY_DT"] = real_date
df3["EXIT_DT"] = real_date


# if ENTRY_TM / EXIT_TM are currently strings like '00:47:26',
# convert them back into full datetimes
df3["ENTRY_TM"] = pd.to_datetime(
   df3["ENTRY_DT"].astype(str) + " " + df3["ENTRY_TM"].astype(str),
   format="%Y-%m-%d %H:%M:%S",
   errors="coerce"
)


df3["EXIT_TM"] = pd.to_datetime(
   df3["EXIT_DT"].astype(str) + " " + df3["EXIT_TM"].astype(str),
   format="%Y-%m-%d %H:%M:%S",
   errors="coerce"
)

In [66]:
df3.head()

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,...,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type,next_orig_lat,next_orig_lon,next_orig_station,walk_distance
0,45.0,100006223599,3968.0,2025-02-12,2025-02-12 06:49:58,2025-02-12,2025-02-12 07:00:33,110715426253,4009.0,0.0,...,BUS,Blk 172,4009.0,1.350814,103.889844,BUS,1.349708,103.873575,Serangoon NEL,195.0
1,NaN,100006223599,109.0,2025-02-12,2025-02-12 07:01:53,2025-02-12,2025-02-12 07:10:18,110713956734,106.0,0.5,...,TRAIN,Serangoon NEL,106.0,1.349708,103.873575,TRAIN,1.319336,103.861570,Boon Keng,0.0
2,NaN,100006223599,106.0,2025-02-12,2025-02-12 15:21:06,2025-02-12,2025-02-12 15:32:56,110713665019,109.0,0.0,...,TRAIN,Boon Keng,109.0,1.319336,103.861570,TRAIN,1.348979,103.872774,S'Goon Stn Exit B,139.0
3,45.0,100006223599,4008.0,2025-02-12,2025-02-12 15:42:15,2025-02-12,2025-02-12 15:53:18,110714355046,3967.0,0.0,...,BUS,S'Goon Stn Exit B,3967.0,1.348979,103.872774,BUS,NaN,NaN,NaN,NaN
4,382.0,130013244516,6927.0,2025-02-12,2025-02-12 08:45:07,2025-02-12,2025-02-12 08:59:01,110713628234,5953.0,0.0,...,BUS,Blk 322 CP,5953.0,1.410690,103.897592,BUS,1.403706,103.902243,Punggol Int B3,0.0


In [67]:
df3.to_pickle('df3.pkl')
print('Saved df3.pkl')

Saved df3.pkl


In [72]:
# memory checkpoint
%who
del df3, real_date, walk_cache, pairs
gc.collect()


consolidated_stations	 df3	 gc	 pairs	 pd	 real_date	 walk_cache	 


7

### for normal laptops: 
restart kernel here and rerun top few cells to get consolidated_stations again

In [ ]:
# # memory checkpoint
# %who
# del bus_additional_data,bus_additional_data_clean,bus_consolidated,bus_data,bus_location_data,bus_needed,bus_needed_final,train_data,train_needed	 

bus_additional_data	 bus_additional_data_clean	 bus_consolidated	 bus_data	 bus_location_data	 bus_needed	 bus_needed_final	 consolidated_stations	 gc	 
pd	 train_data	 train_needed	 


#### pt_jrny (internal validity)

In [8]:
df4 = pd.read_csv('../../data/A0008D - v_pt_jrny_all (full).csv')

In [9]:
df4.shape

(33956202, 17)

In [11]:
df4.columns

Index(['BIZ_DT', 'CRD_NUM', 'JRNY_DEST_ID_NUM', 'JRNY_DIST_KM_CNT',
       'JRNY_END_DT', 'JRNY_END_TM', 'JRNY_FARE_AMT', 'JRNY_ID_NUM',
       'JRNY_ORIG_ID_NUM', 'JRNY_ST_CD', 'JRNY_START_DT', 'JRNY_START_TM',
       'JRNY_TM_MIN_CNT', 'PATRON_CATG_ID_NUM', 'PAY_CD', 'TKT_TYP_CD',
       'TRNSPT_MODE_CD'],
      dtype='object')

In [13]:
df4['JRNY_START_DT'].head()

0    2025-02-16
1    2025-02-16
2    2025-02-16
3    2025-02-16
4    2025-02-16
Name: JRNY_START_DT, dtype: object

In [15]:
# # memory checkpoint - Filtering first
df4 = df4[df4["JRNY_START_DT"].str.startswith("2025-02-12")]
print(df4.shape)
gc.collect()

(2604568, 17)


2315

In [16]:
df4['BIZ_DT'] = pd.to_datetime(df4['BIZ_DT'])
df4['JRNY_START_DT'] = pd.to_datetime(df4['JRNY_START_DT'])
df4['JRNY_END_DT'] = pd.to_datetime(df4['JRNY_END_DT'])
df4['JRNY_START_TM'] = pd.to_datetime(
   df4['JRNY_START_DT'].astype(str) + ' ' + df4['JRNY_START_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
df4['JRNY_END_TM'] = pd.to_datetime(
   df4['JRNY_END_DT'].astype(str) + ' ' + df4['JRNY_END_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)

C:\Users\shaog\AppData\Local\Temp\ipykernel_23340\1308685087.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['BIZ_DT'] = pd.to_datetime(df4['BIZ_DT'])
C:\Users\shaog\AppData\Local\Temp\ipykernel_23340\1308685087.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['JRNY_START_DT'] = pd.to_datetime(df4['JRNY_START_DT'])
C:\Users\shaog\AppData\Local\Temp\ipykernel_23340\1308685087.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_in

In [17]:
df4.dtypes

BIZ_DT                datetime64[ns]
CRD_NUM                        int64
JRNY_DEST_ID_NUM             float64
JRNY_DIST_KM_CNT             float64
JRNY_END_DT           datetime64[ns]
JRNY_END_TM           datetime64[ns]
JRNY_FARE_AMT                float64
JRNY_ID_NUM                    int64
JRNY_ORIG_ID_NUM             float64
JRNY_ST_CD                     int64
JRNY_START_DT         datetime64[ns]
JRNY_START_TM         datetime64[ns]
JRNY_TM_MIN_CNT              float64
PATRON_CATG_ID_NUM             int64
PAY_CD                         int64
TKT_TYP_CD                     int64
TRNSPT_MODE_CD                 int64
dtype: object

In [18]:
df5 = df4[['CRD_NUM', 'JRNY_DEST_ID_NUM', 'JRNY_START_DT',
      'JRNY_START_TM', 'JRNY_END_DT', 'JRNY_END_TM', 'JRNY_ORIG_ID_NUM', 'JRNY_DIST_KM_CNT',
      'JRNY_FARE_AMT', 'JRNY_ID_NUM', 'JRNY_TM_MIN_CNT','PATRON_CATG_ID_NUM','TRNSPT_MODE_CD']]

In [ ]:
# # memory checkpoint
# %who
# del df4
# gc.collect()

consolidated_stations	 df4	 df5	 gc	 pd	 


In [22]:
df5 = df5.merge(consolidated_stations, how = 'left', left_on= 'JRNY_DEST_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [23]:
df5.rename(columns={
   'STATION_NAME': 'DEST_STATION_NAME',
   'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
   'LATITUDE': 'DEST_LATITUDE',
   'LONGITUDE': 'DEST_LONGITUDE',
   'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [24]:
df5 = df5.merge(consolidated_stations, how = 'left', left_on= 'JRNY_ORIG_ID_NUM',
               right_on= 'MRK_ID_NUM')
df5.rename(columns={
   'STATION_NAME': 'ORIG_STATION_NAME',
   'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
   'LATITUDE': 'ORIG_LATITUDE',
   'LONGITUDE': 'ORIG_LONGITUDE',
   'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)

In [53]:
#df5.head()

In [54]:
# %who
# del consolidated_stations

### abt_jrny (internal validity)

In [25]:
abt3 = pd.read_csv('../../data/A0008D - v_abt_pt_jrny (full).csv')

C:\Users\shaog\AppData\Local\Temp\ipykernel_23340\3756125998.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  abt3 = pd.read_csv('../../data/A0008D - v_abt_pt_jrny (full).csv')


In [26]:
abt3.shape

(33924352, 17)

In [28]:
# # memory checkpoint - Filtering first
abt3 = abt3[abt3["JRNY_START_DT"].str.startswith("2025-02-12", na=False)]
print(abt3.shape)
gc.collect()

(2579117, 17)


606

In [30]:
abt3['BIZ_DT'] = pd.to_datetime(abt3['BIZ_DT'])
abt3['JRNY_START_DT'] = pd.to_datetime(abt3['JRNY_START_DT'])
abt3['JRNY_END_DT'] = pd.to_datetime(abt3['JRNY_END_DT'])
abt3['JRNY_START_TM'] = pd.to_datetime(
   abt3['JRNY_START_DT'].astype(str) + ' ' + abt3['JRNY_START_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)
abt3['JRNY_END_TM'] = pd.to_datetime(
   abt3['JRNY_END_DT'].astype(str) + ' ' + abt3['JRNY_END_TM'],
   format='%Y-%m-%d %H:%M:%S.%f'
)

C:\Users\shaog\AppData\Local\Temp\ipykernel_23340\3513173403.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  abt3['BIZ_DT'] = pd.to_datetime(abt3['BIZ_DT'])
C:\Users\shaog\AppData\Local\Temp\ipykernel_23340\3513173403.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  abt3['JRNY_START_DT'] = pd.to_datetime(abt3['JRNY_START_DT'])
C:\Users\shaog\AppData\Local\Temp\ipykernel_23340\3513173403.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[ro

In [57]:
abt3.dtypes

BIZ_DT                datetime64[us]
CRD_NUM                       object
JRNY_DEST_ID_NUM             float64
JRNY_DIST_KM_CNT             float64
JRNY_END_DT           datetime64[us]
JRNY_END_TM           datetime64[us]
JRNY_FARE_AMT                float64
JRNY_ID_NUM                      str
JRNY_ORIG_ID_NUM             float64
JRNY_ST_CD                     int64
JRNY_START_DT         datetime64[us]
JRNY_START_TM         datetime64[us]
JRNY_TM_MIN_CNT              float64
PATRON_CATG_ID_NUM           float64
PAY_CD                       float64
TKT_TYP_CD                   float64
TRNSPT_MODE_CD                 int64
dtype: object

In [31]:
abt4 = abt3[['CRD_NUM', 'JRNY_DEST_ID_NUM', 'JRNY_START_DT',
      'JRNY_START_TM', 'JRNY_END_DT', 'JRNY_END_TM', 'JRNY_ORIG_ID_NUM', 'JRNY_DIST_KM_CNT',
      'JRNY_FARE_AMT', 'JRNY_ID_NUM', 'JRNY_TM_MIN_CNT','PATRON_CATG_ID_NUM','TRNSPT_MODE_CD']]

In [32]:
abt4 = abt4.merge(consolidated_stations, how = 'left', left_on= 'JRNY_DEST_ID_NUM',
               right_on= 'MRK_ID_NUM')

In [33]:
abt4.rename(columns={
   'STATION_NAME': 'DEST_STATION_NAME',
   'MRK_ID_NUM': 'DEST_MRK_ID_NUM',
   'LATITUDE': 'DEST_LATITUDE',
   'LONGITUDE': 'DEST_LONGITUDE',
   'Travel_Type': 'DEST_Travel_Type'
}, inplace=True)

In [34]:
abt4 = abt4.merge(consolidated_stations, how = 'left', left_on= 'JRNY_ORIG_ID_NUM',
               right_on= 'MRK_ID_NUM')
abt4.rename(columns={
   'STATION_NAME': 'ORIG_STATION_NAME',
   'MRK_ID_NUM': 'ORIG_MRK_ID_NUM',
   'LATITUDE': 'ORIG_LATITUDE',
   'LONGITUDE': 'ORIG_LONGITUDE',
   'Travel_Type': 'ORIG_Travel_Type'
}, inplace=True)

In [ ]:
# # memory checkpoint
# %who
# del abt3

abt3	 abt4	 consolidated_stations	 df5	 gc	 pd	 


### combine pt_jrny and abt_jrny for internal validity

In [37]:
df5['CRD_NUM'] = df5['CRD_NUM'].astype('object')

In [38]:
df5 = pd.concat(
   [df5, abt4],
   axis=0,
   ignore_index=True
)
df5 = df5.sort_values(["CRD_NUM", "JRNY_START_DT", "JRNY_START_TM"], ascending= [True, True, True]).reset_index(drop= True)

In [39]:
df5 = df5[df5["JRNY_START_DT"].dt.date == pd.Timestamp("2025-02-12").date()]

In [40]:
df5.head()

,CRD_NUM,JRNY_DEST_ID_NUM,JRNY_START_DT,JRNY_START_TM,JRNY_END_DT,JRNY_END_TM,JRNY_ORIG_ID_NUM,JRNY_DIST_KM_CNT,JRNY_FARE_AMT,JRNY_ID_NUM,...,DEST_STATION_NAME,DEST_MRK_ID_NUM,DEST_LATITUDE,DEST_LONGITUDE,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type
0,9999,NaN,2025-02-12,2025-02-12 00:00:35,2025-02-12,2025-02-12 00:00:35,4413.0,NaN,1.9,110709304910,...,Blk 308A,NaN,1.361836,103.738264,BUS,Comfort Driving Ctr,4413.0,1.335546,103.898253,BUS
1,9999,NaN,2025-02-12,2025-02-12 00:00:35,2025-02-12,2025-02-12 00:00:35,4413.0,NaN,1.9,110709304910,...,Blk 319A,NaN,1.361852,103.738391,BUS,Comfort Driving Ctr,4413.0,1.335546,103.898253,BUS
2,9999,NaN,2025-02-12,2025-02-12 00:00:35,2025-02-12,2025-02-12 00:00:35,4413.0,NaN,1.9,110709304910,...,Blk 126A,NaN,1.359326,103.736336,BUS,Comfort Driving Ctr,4413.0,1.335546,103.898253,BUS
3,9999,NaN,2025-02-12,2025-02-12 00:00:35,2025-02-12,2025-02-12 00:00:35,4413.0,NaN,1.9,110709304910,...,Blk 307B,NaN,1.359592,103.736422,BUS,Comfort Driving Ctr,4413.0,1.335546,103.898253,BUS
4,9999,NaN,2025-02-12,2025-02-12 00:00:35,2025-02-12,2025-02-12 00:00:35,4413.0,NaN,1.9,110709304910,...,Blk 962A,NaN,1.339649,103.938065,BUS,Comfort Driving Ctr,4413.0,1.335546,103.898253,BUS


In [41]:
df5.to_pickle('df5.pkl')
print('Saved df5.pkl')

Saved df5.pkl
